# All-in-one real-world multimodal demo

Recommended recording notebook. It runs: setup -> real yfinance OHLCV -> features/labels -> aligned multimodal NPZ -> ablations -> diagnostics -> embedding visuals.

See `docs/architecture.md` for model details. This demo validates the multimodal representation pipeline and evaluation harness. It is not a portfolio backtest or trading system.


In [ ]:
import os, subprocess, sys
from pathlib import Path
from datetime import date, timedelta

REPO_URL = "https://github.com/Randhir123/nifty50-multimodal-transformer.git"
REPO_DIR = "/content/nifty50-multimodal-transformer"
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
subprocess.run(["git", "pull", "origin", "main"], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
print("Ready:", os.getcwd())


## Step 1: Download real OHLCV data

Download stock and benchmark price history from yfinance and cache as CSV. All subsequent steps read from these local files so the download only runs once per session.


In [ ]:
import os, subprocess, sys
from pathlib import Path
from datetime import date, timedelta
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

from src.data.download_yfinance import download_multiple_tickers, download_benchmark_data, save_ticker_csv

TICKERS = ["RELIANCE.NS", "TCS.NS", "INFY.NS"]
BENCHMARK = "^NSEI"
SECTORS = {"RELIANCE.NS": "Energy", "TCS.NS": "IT", "INFY.NS": "IT"}
START = (date.today() - timedelta(days=31 * 9)).isoformat()
END = date.today().isoformat()
OUTPUT_DIR = Path("data/processed/real_world_demo")
RAW_DIR = OUTPUT_DIR / "raw"
CHART_DIR = OUTPUT_DIR / "charts"
for p in [OUTPUT_DIR, RAW_DIR, CHART_DIR]:
    p.mkdir(parents=True, exist_ok=True)

WINDOW_SIZE = 20
HORIZON_DAYS = 3
ABLATION_EPOCHS = 10
ABLATION_BATCH_SIZE = 16
FEATURE_COLUMNS = [
    "log_return_1d", "cum_return_3d", "cum_return_5d", "cum_return_10d",
    "realized_vol_5d", "realized_vol_10d", "high_low_range_over_close",
    "close_over_10dma_minus_1", "close_over_20dma_minus_1",
    "volume_over_20d_avg", "stock_minus_index_return",
]

print(f"Downloading {TICKERS}  |  {START} to {END}")
stock_data = download_multiple_tickers(TICKERS, start=START, end=END)
for ticker, df in stock_data.items():
    path = save_ticker_csv(ticker, df, output_dir=RAW_DIR)
    print(f"  {ticker}: {df.shape}  -> {path}")
benchmark_df = download_benchmark_data(BENCHMARK, start=START, end=END)
save_ticker_csv(BENCHMARK, benchmark_df, output_dir=RAW_DIR)
print(f"  {BENCHMARK}: {benchmark_df.shape}")


## Step 2: Feature engineering and labels

Compute 11 OHLCV-derived technical features and the future benchmark-relative outperformance label for each stock/date row.

The label is `1` when the stock outperforms Nifty over the next `HORIZON_DAYS` trading days, `0` otherwise. Labels are the supervised target — never included in input tensors.


In [ ]:
from src.data.features import compute_technical_features
from src.data.labels import generate_outperformance_label

frames = []
for ticker in TICKERS:
    features = compute_technical_features(stock_data[ticker], benchmark_df)
    labelled = generate_outperformance_label(features, horizon_days=HORIZON_DAYS)
    labelled["stock_id"] = ticker
    frames.append(labelled)

combined = pd.concat(frames, ignore_index=True)
required = ["stock_id", "date", "label", *FEATURE_COLUMNS]
tabular_df = (
    combined.dropna(subset=required)
    .loc[:, required + ["close", "volume"]]
    .reset_index(drop=True)
)
tabular_df.to_csv(OUTPUT_DIR / "tabular_samples.csv", index=False)

display(tabular_df.head())
display(
    tabular_df.groupby("stock_id")["label"]
    .agg(["count", "sum", "mean"])
    .rename(columns={"sum": "positive_labels", "mean": "positive_rate"})
)


## Step 3: Text records and KG context

Build leakage-safe text records from real OHLCV-derived market summaries. Each record has an `event_date <= end_date` cutoff so no future information leaks into model inputs.

The KG context adds sector mappings, recent-return features, and high-volume event flags.


In [ ]:
from src.data.multimodal_samples import build_tabular_multimodal_samples

arrays = build_tabular_multimodal_samples(tabular_df, feature_cols=FEATURE_COLUMNS, window_size=WINDOW_SIZE)

# Leakage-safe text records derived from real OHLCV features
records = []
for ticker, frame in tabular_df.groupby("stock_id"):
    frame = frame.sort_values("date").reset_index(drop=True)
    for idx in range(0, len(frame), 5):
        row = frame.iloc[idx]
        direction = "positive" if row["log_return_1d"] >= 0 else "negative"
        records.append({
            "stock_id": ticker,
            "event_date": row["date"],
            "source_type": "market_summary",
            "title": f"{ticker} {direction} daily market summary",
            "body_text": (
                f"As of {row['date'].date()}, {ticker} had a {direction} one-day return of "
                f"{row['log_return_1d']:.4f}, relative return versus index of "
                f"{row['stock_minus_index_return']:.4f}, and volume ratio {row['volume_over_20d_avg']:.4f}."
            ),
        })
text_records = pd.DataFrame(records)
text_records.to_csv(OUTPUT_DIR / "text_records.csv", index=False)
print(f"Text records: {len(text_records)} rows")

# KG: recent returns
kg_returns = tabular_df[["stock_id", "date", "stock_minus_index_return"]].rename(
    columns={"stock_minus_index_return": "recent_return"}
)
kg_returns.to_csv(OUTPUT_DIR / "kg_returns.csv", index=False)

# KG: high-volume event flags
events = []
for ticker, frame in tabular_df.groupby("stock_id"):
    threshold = frame["volume_over_20d_avg"].quantile(0.9)
    for _, row in frame.loc[frame["volume_over_20d_avg"] >= threshold].iterrows():
        events.append({"stock_id": ticker, "event_date": row["date"], "event_type": "high_volume"})
event_records = (
    pd.DataFrame(events) if events
    else pd.DataFrame(columns=["stock_id", "event_date", "event_type"])
)
event_records.to_csv(OUTPUT_DIR / "event_records.csv", index=False)
print(f"High-volume event flags: {len(event_records)}")

sector_df = pd.DataFrame([{"stock_id": k, "sector_id": v} for k, v in SECTORS.items()])
sector_df.to_csv(OUTPUT_DIR / "stock_sectors.csv", index=False)
display(sector_df)


## Step 4: Generate candlestick chart images

Generate a candlestick chart PNG for each aligned `(stock_id, end_date)` sample. Each chart is not decorative — it is one modality token in the training artifact.


In [ ]:
from src.viz.charts import generate_or_resolve_sample_chart

for stock_id, end_date in zip(arrays.stock_ids, arrays.end_dates):
    generate_or_resolve_sample_chart(
        stock_data[str(stock_id)],
        symbol=str(stock_id),
        prediction_date=pd.Timestamp(end_date),
        output_dir=CHART_DIR,
        lookback_days=20,
    )

chart_files = sorted(CHART_DIR.glob("*.png"))
print(f"Generated {len(chart_files)} candlestick charts in {CHART_DIR}")

# Preview the first three
for path in chart_files[:3]:
    print(path.name)
    display(Image(filename=str(path)))


## Step 5: Attach modality tokens and save the aligned NPZ

This is the core alignment step. Every modality is keyed by the same `(stock_id, end_date)` pair:

| Key | Shape | Description |
|---|---|---|
| `tabular_tokens` | `[N, 20, 11]` | rolling OHLCV feature window |
| `image_tokens` | `[N, 16]` | CLS embedding from ImageTransformer |
| `text_tokens` | `[N, 16]` | deterministic text hash vector |
| `kg_tokens` | `[N, ~4]` | sector/peer/event context |
| `y` | `[N]` | outperformance label |

The FusionTransformer projects all modalities to a common 16-dimensional space and applies self-attention across them.


In [ ]:
from src.data.multimodal_samples import attach_text_tokens, attach_kg_tokens, attach_image_tokens, save_multimodal_samples
from src.kg.build_graph import build_market_knowledge_graph
from src.models.image_transformer import ImageTransformerConfig

graph = build_market_knowledge_graph(SECTORS, event_records=event_records)
arrays = attach_text_tokens(arrays, text_records, dim=16)
arrays = attach_kg_tokens(arrays, graph, returns=kg_returns)
arrays = attach_image_tokens(
    arrays,
    chart_dir=CHART_DIR,
    config=ImageTransformerConfig(image_size=64, patch_size=16, model_dim=16, num_heads=4, num_layers=1, ff_dim=32),
    device="cpu",
)
dataset_path = save_multimodal_samples(arrays, OUTPUT_DIR / "real_world_multimodal_samples.npz")
print(f"Saved: {dataset_path}")

# Inspect actual tensor shapes
data = dict(np.load(dataset_path, allow_pickle=False))
print("\nAligned multimodal artifact shapes:")
for key, arr in data.items():
    print(f"  {key:16s}  {str(arr.shape):20s}  {arr.dtype}")


## Ablations and diagnostics

The run uses 10 epochs by default. Interpret the output through diagnostics: majority baseline, predicted-positive rate, and ROC-AUC. Do not treat these compact metrics as backtest performance.


In [ ]:
ABLATION_DIR = OUTPUT_DIR / "ablations"
cmd = [sys.executable, "scripts/run_ablation_study.py", "--dataset", str(dataset_path), "--output-dir", str(ABLATION_DIR), "--epochs", str(ABLATION_EPOCHS), "--batch-size", str(ABLATION_BATCH_SIZE), "--device", "cpu", "--model-dim", "16", "--num-heads", "4", "--num-layers", "1", "--ff-dim", "32", "--val-fraction", "0.25"]
print(" ".join(cmd))
subprocess.run(cmd, check=True)
results = pd.read_csv(ABLATION_DIR / "ablation_results.csv")
display(results)

print((ABLATION_DIR / "ablation_diagnostics.md").read_text(encoding="utf-8"))
summary_cols = ["variant","accuracy","majority_class_accuracy","predicted_positive_rate","f1","roc_auc","probability_mean"]
display(results[summary_cols])
plot_df = results.set_index("variant")
for cols, title, outfile in [(["accuracy","majority_class_accuracy"], "Accuracy vs majority-class baseline", "accuracy_vs_majority_baseline.png"), (["roc_auc"], "ROC-AUC by modality combination", "roc_auc_by_variant.png"), (["val_positive_rate","predicted_positive_rate"], "Actual validation positive rate vs predicted positive rate", "positive_rate_diagnostics.png")]:
    ax = plot_df[cols].plot(kind="bar", figsize=(11,4))
    ax.set_title(title); ax.set_ylim(0,1); plt.xticks(rotation=30, ha="right"); plt.tight_layout(); plt.savefig(OUTPUT_DIR / outfile, dpi=160); plt.show()


## Actual embedding visualization

This PCA plot is generated from the real multimodal artifact, not from conceptual slides.


In [ ]:
from sklearn.decomposition import PCA
modalities = {"tabular": data["tabular_tokens"].mean(axis=1), "image": data["image_tokens"], "text": data["text_tokens"], "kg": data["kg_tokens"]}
rows=[]
for modality, values in modalities.items():
    n = 2 if values.shape[1] >= 2 else 1
    projection = PCA(n_components=n).fit_transform(values)
    if n == 1: projection = np.c_[projection[:,0], np.zeros(values.shape[0])]
    for i, (x, y) in enumerate(projection):
        rows.append({"sample_id":i,"stock_id":str(data["stock_ids"][i]),"end_date":str(data["end_dates"][i]),"label":int(data["y"][i]),"modality":modality,"x":float(x),"y":float(y)})
projection_df = pd.DataFrame(rows)
projection_df.to_csv(OUTPUT_DIR / "modality_embedding_projection.csv", index=False)
fig, ax = plt.subplots(figsize=(10,7))
for modality, frame in projection_df.groupby("modality"):
    ax.scatter(frame["x"], frame["y"], label=modality, alpha=0.75)
ax.set_title("Actual modality embedding projections"); ax.set_xlabel("PCA 1"); ax.set_ylabel("PCA 2"); ax.legend(); plt.tight_layout(); plt.savefig(OUTPUT_DIR / "modality_embedding_projection.png", dpi=160); plt.show()


In [ ]:
gallery_path = OUTPUT_DIR / "sample_gallery.md"
lines = ["# Real Demo Sample Gallery", ""]
for i in range(min(10, len(data["stock_ids"]))):
    stock_id = str(data["stock_ids"][i]); end_date = pd.Timestamp(data["end_dates"][i]).strftime("%Y%m%d"); label = int(data["y"][i]); chart_name = f"{stock_id}_{end_date}.png"
    lines += [f"## Sample {i}: {stock_id} / {end_date}", "", f"- label: `{label}`", f"- chart: `charts/{chart_name}`", "", f"![{chart_name}](charts/{chart_name})", ""]
gallery_path.write_text("\n".join(lines), encoding="utf-8")
summary_path = OUTPUT_DIR / "demo_visual_summary.md"
summary_path.write_text("\n".join(["# Demo Visual Summary", "", "Generated from actual real-world demo artifacts.", "", "## Key outputs", "", "- `real_world_multimodal_samples.npz`", "- `ablation_results.csv`", "- `ablation_diagnostics.md`", "- `accuracy_vs_majority_baseline.png`", "- `roc_auc_by_variant.png`", "- `positive_rate_diagnostics.png`", "- `modality_embedding_projection.png`", "- `sample_gallery.md`", "", "## Recording message", "", "This notebook demonstrates the full multimodal representation pipeline and evaluation harness. Compare accuracy with the majority baseline, inspect predicted-positive rate, and avoid presenting compact demo metrics as portfolio backtest performance."]), encoding="utf-8")
print("Wrote", gallery_path)
print("Wrote", summary_path)
print("Output directory", OUTPUT_DIR)


## Final recording checklist

Use these outputs for the demo:

```text
real_world_multimodal_samples.npz
ablation_results.csv
ablation_diagnostics.md
prediction_scores_<variant>.csv
accuracy_vs_majority_baseline.png
roc_auc_by_variant.png
positive_rate_diagnostics.png
modality_embedding_projection.png
sample_gallery.md
demo_visual_summary.md
```

The safe claim is: the notebook demonstrates the multimodal pipeline and evaluation harness. It does not yet prove investment performance.
